# Masked Diffusion Language Models: Step-by-Step Walkthrough

This notebook summarizes the training and sampling pipeline used in the repository implementation of **Simple and Effective Masked Diffusion Language Models (MDLM)**. Each section mirrors the code structure and equations visible in the source, so that readers can map the method to implementation details.

## 1. Noise schedule

MDLM supports several schedules for the continuous-time noise level $\sigma(t)$. The default `loglinear` schedule sets

$$\sigma(t) = -\log\big(1 - (1 - \varepsilon) t\big),$$

which grows smoothly from $\sigma_{\min} = \varepsilon$ to $\sigma_{\max} \approx 1$ as $t \in [0,1]$ (see `noise_schedule.LogLinearNoise.total_noise`). The rate term used in the diffusion SDE is

$$g(t) = \frac{\mathrm{d}\sigma(t)}{\mathrm{d} t} = \frac{1 - \varepsilon}{1 - (1 - \varepsilon) t},$$

matching `rate_noise`. These terms feed the forward and reverse processes described below.

In [ ]:
import torch
from noise_schedule import LogLinearNoise

noise = LogLinearNoise(eps=1e-3)
ts = torch.linspace(0, 1, 5)
print('sigma(t):', noise.total_noise(ts))
print('g(t):', noise.rate_noise(ts))

## 2. Forward noising (masking) process

The forward diffusion replaces tokens with the mask token with probability $\text{move\_chance}(t)$. In continuous time, when using `loglinear` noise, the default masking probability is

$$\text{move\_chance}(t) = 1 - e^{-\sigma(t)}.$$

Given clean tokens $x_0$, the noisy sample $x_t$ is drawn as

$$x_t = \begin{cases}\text{[MASK]} & \text{with probability } \text{move\_chance}(t),\\ x_0 & \text{otherwise.}\end{cases}$$

This is implemented in `Diffusion.q_xt`, where Bernoulli draws decide whether each position is replaced by the mask index.

In [ ]:
import torch

def q_xt(x0, move_chance, mask_index):
    move_indices = torch.rand_like(x0, dtype=torch.float32) < move_chance
    return torch.where(move_indices, torch.full_like(x0, mask_index), x0)

x0 = torch.tensor([[1, 2, 3, 4]])
mask_index = 99
move = torch.tensor([[0.2]])  # 20% chance to mask each token
xt = q_xt(x0, move, mask_index)
xt

## 3. Parameterizations of the reverse model

The denoising network outputs per-token logits; different parameterizations map logits to log-probabilities $\log p_\theta(x_0 \mid x_t, t)$:

### SUBS (Substitution) parameterization
* Mask logits are forced to $-\infty$ so probability mass only lies on real tokens.
* Observed (unmasked) tokens are clamped to a delta distribution: logits for their ground-truth index are set to 0 and all others to $-\infty$.
* The logits are normalized with `logsumexp` to represent log-probabilities.

Formally, for each position $i$ with input $x_t^i$, the transformed logits $\tilde{\ell}$ satisfy

$$\tilde{\ell}^i_k = \begin{cases}-\infty & k = \text{[MASK]} \\n0 & x_t^i = k \\n-\infty & x_t^i \neq k\end{cases}$$

### SEDD-style parameterization
* Scales logits by $\log( e^{\sigma(t)} - 1 )$ and allocates uniform mass over non-input tokens, then sets the log-score for the current input token to 0:

$$\tilde{\ell} = \ell - \log\big(e^{\sigma(t)} - 1\big) - \log(V-1), \quad \tilde{\ell}_{x_t} = 0.$$

### D3PM parameterization
* Normalizes logits (optionally masking the [MASK] token) with `logsumexp` to obtain a categorical log-probability distribution used in the variational bound.

In [ ]:
import torch

def subs_parameterization(logits, xt, mask_index, neg_inf=-1e6):
    logits = logits.clone()
    logits[:, :, mask_index] += neg_inf
    logits = logits - torch.logsumexp(logits, dim=-1, keepdim=True)
    unmasked = xt != mask_index
    logits[unmasked] = neg_inf
    logits[unmasked, xt[unmasked]] = 0
    return logits

logits = torch.randn(1, 3, 5)
xt = torch.tensor([[4, 99, 2]])
logits_subs = subs_parameterization(logits, xt, mask_index=99)
logits_subs.exp().sum(-1)  # probabilities sum to 1 per position

## 4. Training losses

### Discrete-time D3PM loss
For a discretized schedule with $T$ steps, the code computes a variational bound term (implemented in `_d3pm_loss`) that mixes two KL-like components weighted by $\tfrac{\Delta t}{t}$ and $1 - \tfrac{\Delta t}{t}$, evaluated only on masked positions $x_t = \text{[MASK]}$. A reconstruction cross-entropy at $t=0$ is optionally added for the D3PM parameterization.

### Continuous-time SUBS objective
When using SUBS in continuous time, the objective reduces to the negative log-probability of the clean token under the predicted distribution scaled by the noise rate:

$$\mathcal{L}_\text{SUBS} = -\log p_\theta(x_0 \mid x_t, t)\; \frac{g(t)}{e^{\sigma(t)} - 1}.$$

If importance sampling or a change of variables is enabled, the scaling is adjusted accordingly (see `_forward_pass_diffusion`).

### SEDD score-entropy objective
SEDD uses the entropy-based loss (`_score_entropy`) restricted to masked positions, combining positive mass on alternative tokens with a negative term on the ground-truth token and a constant derived from the forward masking probability.

In [ ]:
import torch

def subs_continuous_loss(log_p_theta, dsigma, sigma):
    return -log_p_theta * (dsigma / torch.expm1(sigma))

log_p_theta = torch.log(torch.tensor([[0.8, 0.9]]))  # dummy per-token probs
sigma = torch.tensor([0.5])
_, dsigma = noise(sigma)  # reuse noise from earlier cell
loss_vals = subs_continuous_loss(log_p_theta, dsigma[:, None], sigma[:, None])
loss_vals

## 5. Reverse sampling updates

Ancestral reverse diffusion (`_ddpm_update`) samples $x_{t-\Delta t}$ from $x_t$ by mixing the model's predicted $p_\theta(x_0 \mid x_t, t)$ with the forward transition probabilities:

1. Compute $\text{move\_chance}(t) = 1 - e^{-\sigma(t)}$ and $\text{move\_chance}(t-\Delta t)$.
2. Form $q(x_{t-\Delta t} \mid x_t)$ proportional to $(\text{move\_chance}(t) - \text{move\_chance}(t-\Delta t)) p_\theta(x_0 \mid x_t, t)$ with mass $\text{move\_chance}(t-\Delta t)$ on the [MASK] token.
3. Draw the next token from this categorical distribution, copying already unmasked tokens to avoid overwriting observed inputs.

The cached variant (`_ddpm_caching_update`) reuses $p_\theta(x_0 \mid x_t, t)$ across steps when possible to accelerate sampling.

In [ ]:
def ddpm_step(x, t, dt, mask_index, model_log_probs):
    sigma_t = noise.total_noise(t)
    sigma_s = noise.total_noise(t - dt)
    move_t = (1 - torch.exp(-sigma_t))[:, None, None]
    move_s = (1 - torch.exp(-sigma_s))[:, None, None]
    q_xs = model_log_probs.exp() * (move_t - move_s)
    q_xs[:, :, mask_index] = move_s[:, :, 0]
    # Sample category
    gumbel_norm = (1e-10 - (torch.rand_like(q_xs) + 1e-10).log())
    x_next = (q_xs / gumbel_norm).argmax(dim=-1)
    copy_flag = (x != mask_index).to(x.dtype)
    return copy_flag * x + (1 - copy_flag) * x_next

# Dummy single-step illustration
x = torch.full((1, 4), mask_index)
model_log_probs = torch.log_softmax(torch.randn(1, 4, 5), dim=-1)
x_prev = ddpm_step(x, t=torch.tensor([1.0]), dt=torch.tensor([0.1]), mask_index=mask_index, model_log_probs=model_log_probs)
x_prev

## 6. Putting it together

A full training step proceeds by:
1. Sampling a random time $t$ (optionally discretized to $T$ steps) and computing $\sigma(t)$ and $g(t)$.
2. Masking tokens to obtain $x_t \sim q(x_t \mid x_0, t)$ using the move probability.
3. Running the denoiser to obtain parameterized log-probabilities $p_\theta(x_0 \mid x_t, t)$.
4. Applying the chosen loss (SUBS, SEDD, or D3PM) and backpropagating.

During inference, the sampler iteratively applies the reverse update from a fully masked prior until $t=0$, optionally reusing cached predictions for efficiency.